# Prognose des Oberschwingungsreststroms – Netzmodell einer Sammelschiene

Ein **optionaler Teil** von groundfault: Er baut auf den **Ergebnisdateien** der Einzelauswertungen auf
(`Messung_auswerten.ipynb`, Abschnitt 7) und braucht sonst nichts von der Auswertung. Das **Netzmodell** der
Erdschlussschleife an einer Sammelschiene je Oberschwingungsordnung wird mit den Erdschlussversuchen kalibriert, mit
jedem an der Sammelschiene aufgezeichneten Erdschluss nachgeführt und geprüft; aus Oberschwingungsspannungen – z. B.
den 95-%-Quantilen einer Netzqualitätsmessung – prognostiziert es den **Oberschwingungsreststrom** I_H und den
Reststrom I_RES eines Erdschlusses an der Sammelschiene oder hinter einer Leitung. Ausgangsfassung:
`Vorlagen/Netzmodell_Prognose_DEFAULT.ipynb`. Verfahren und Ersatzschaltbilder: Online-Dokumentation, *Forecast*.

1. **Kalibrierung** mit den Erdschlussversuchen der Kampagnen (Fehlerstrom gemessen): Sequenzimpedanzen Z_0, Z_1, Z_2
   je Ordnung aus den Änderungen der Sammelschienenspannungen, Z_q = (Z_0 + Z_1 + Z_2)/3; bei den Ordnungen des
   Nullsystems zusätzlich die Quelle, die nur im Fehler besteht (sättigende Spulen und Wandler).
2. **Prüfung zwischen den Versuchen** – die Abweichungen zeigen, welchen Zuschlag eine Prognose braucht.
3. **Aufgezeichnete Erdschlüsse** (Sammelschienenspannungen, z. B. eines Störschreibers): Z_1, Z_2, Z_q nachgeführt,
   der Fehlerstrom geschätzt (mit Spulenströmen über die Knotenregel bei jeder Ordnung) und die Prognose an jedem von
   ihnen geprüft.
4. **Prüfung mit weiteren Messungen**, z. B. einem Erdschlussversuch an einem Ort im Netz.
5. **Prognose** von I_H und I_RES (ein Wert je Ordnung oder eine Zeitreihe; an der Sammelschiene oder hinter `LINE`).
6. Modelldatei und Tabellen im Prognoseordner des Arbeitsordners **speichern**.

In [ ]:
%matplotlib inline
import datetime
import json
import sys
import warnings
from pathlib import Path

LANGUAGE = "de"                # Sprache von Tabellen, Hinweisen und Abbildungen: "de" | "en"
try:
    import groundfault as gf
except ImportError:            # nicht installiert: Quellordner eines Klons verwenden (…/groundfault/src)
    for p in [Path.cwd(), *Path.cwd().parents]:    # ./src, ./groundfault/src, ./*/groundfault/src
        try:
            candidates = [p / "src", p / "groundfault" / "src", *sorted(p.glob("*/groundfault/src"))]
        except OSError:
            candidates = [p / "src", p / "groundfault" / "src"]
        hits = [c for c in candidates if (c / "groundfault" / "__init__.py").is_file()]
        if hits:
            sys.path.insert(0, str(hits[0]))
            break
    import groundfault as gf
import numpy as np
import pandas as pd
from IPython.display import Markdown, display
from groundfault import forecast as fc
from groundfault import (campaign, corrections, demo, diagnostics, disturbances, document, earthing,
                         equivalent_source, explorer, i18n, interactive, layout, legacy, pq, register, report, rms, sites,
                         transformer)

i18n.set_language(LANGUAGE)
# Arbeitsordner = dieser Ordner oder der erste Ordner darüber mit einem Kampagnenordner („01_Messkampagnen“
# oder „campaigns“); dessen Name gibt die Ordnerstruktur, ohne ihn gilt die Ordnerstruktur von LANGUAGE
WORK_FOLDER, LAYOUT = layout.find_work_folder(Path.cwd(), default=LANGUAGE)
WORK_FOLDER = Path(WORK_FOLDER)
pd.set_option("display.max_columns", 60); pd.set_option("display.width", 220)
pd.set_option("display.float_format", lambda x: "%.4g" % x)
report.activate_style()        # Vorschau im Notebook = Schriftbild der exportierten Abbildungen


def show(df):
    """Tabelle mit Spaltennamen und Codes in der Ausgabesprache."""
    display(i18n.localize(df))


print("groundfault %s · Arbeitsordner: %s · Ordnerstruktur: %s · Schrift der Abbildungen: %s"
      % (gf.__version__, WORK_FOLDER, LAYOUT.language, report.used_font()))

## 1 · Konfiguration

In [ ]:
NETWORK   = "<UW>"                       # Name des Netzmodells (Umspannwerk, Sammelschiene)
DEMO      = False
CAMPAIGNS = ["JJJJ-MM-TT_Ort"]           # Kampagnen, deren Erdschlussversuche das Modell kalibrieren (aus dem Messregister)
TESTS     = None                         # None = die Versuche von CAMPAIGNS aus dem Messregister; oder eine Liste dict(campaign=…, measurement=…, Z=…)
# Aufgezeichnete Erdschlüsse an der Sammelschiene (Nachführung): Ergebnisdateien aus Messung_auswerten.ipynb mit den
# Sammelschienenspannungen vor und im Fehler – ohne Fehlerstrom; mit Spulenströmen auch der Fehlerstrom je Ordnung
FAULTS = [
    # dict(campaign="JJJJ-MM-TT_Ort", measurement="<Aufzeichnung>"),
]
# Messungen zur Prüfung des Modells, z. B. ein Erdschlussversuch an einem Ort im Netz:
# dict(campaign=…, measurement=<Aufzeichnung der Sammelschiene>, Z=…, current=<Aufzeichnung mit dem Fehlerstrom>, line=LINE)
CHECKS = []
# optional je Eintrag: name="…" (Standard: die Messung), meta=dict(…) (z. B. Last, PV-Einspeisung, Schaltzustand),
# transformer="…" (Trafoaufzeichnung desselben Erdschlusses, zweites Messgerät: deren Vorimpedanz Z_vor),
# line=dict(…) (ein Versuch im Netz: die Leitung bis zu seinem Fehlerort)

# ---------------------------------------------------------------- Modell
ORDERS        = "auto"            # "auto" = ungerade Ordnungen ≥ 3 mit mindestens MIN_CURRENT_A im Versuch, sonst z. B. [3, 5, 7, 11, 13]
MIN_CURRENT_A = 0.3               # kleinster Strom einer Ordnung für "auto" in A
HOW           = "median"          # Z_q über die Erdschlüsse: "median" | "mean" (Mittel) | "latest" (der zuletzt hinzugefügte)
TWO_SOURCE    = "zero_sequence"   # Ordnungen mit einer im Fehler entstehenden Quelle: "zero_sequence" (3, 9, 15 …) | [Ordnungen] | None

# ---------------------------------------------------------------- Prognose
VOLTAGES          = None   # Oberschwingungsspannungen je Ordnung in V, z. B. {5: U_5, 7: U_7}, oder eine CSV-Datei (None = keine Prognose)
VOLTAGE_REFERENCE = "LE"          # "LE" = Leiter-Erde | "LL" = Leiter-Leiter (durch √3 geteilt; ohne Nullsystem)
CSV_OPTIONS       = dict(sep=";", decimal=",", index_col=0)   # Lesen einer CSV-Datei (pandas.read_csv): eine Spalte je Ordnung
QUANTILE          = 0.95          # Quantil einer Zeitreihe (und der 10-min-Werte von PQ_FILES)
PQ_FILES          = None          # Export(e) eines PQ-Messgeräts an der Sammelschiene → VOLTAGES = Quantil je Ordnung (größte Schleife/√3 bzw. größter Leiter Leiter-Erde) × Faktor; relativ zum Arbeitsordner oder absolut
PQ_TIMEZONE       = None          # Zeitzone der Geräteuhr, z. B. "Europe/Berlin" (None = Zeitstempel wie exportiert)
PQ_SERIES         = False         # True: jeder 10-min-Wert von PQ_FILES (Zeitreihe) → Quantil QUANTILE von I_H und I_RES
Z_F               = 0.0           # Fehlerimpedanz in Ω (0 = Fehler ohne Widerstand)
LINE              = None   # Fehlerort im Netz: dict(name=…, segments=[dict(length_km=…, R1=…, X1=…, R0=…, X0=…, C1_uF=…)])
# Beläge bei 50 Hz (Ω/km, µF/km) heißen R konstant, X ∝ m – nicht auf der sicheren Seite; für die wichtigen Ordnungen
# eigene Werte angeben: je Abschnitt orders={5: dict(R1=…, X1=…, R0=…, X0=…)} (Ω/km bei 250 Hz, X nicht durch 5 geteilt)
# oder für die ganze Leitung loop={5: Z_qL} (Ω, z. B. gemessen mit MODEL.line_from_fault(<Erdschluss hinter der Leitung>))

# ---------------------------------------------------------------- Zuschläge und Grenzwerte – selbst festlegen
# groundfault enthält keine Zuschläge oder Grenzwerte. Nicht gesetzt (die Werte unten), zeigt die Prognose sie nicht.
MARGIN_A      = 0.0     # Zuschlag je Ordnung in A: eine Zahl oder {Ordnung: A} (0.0 = keiner), z. B. aus der Prüfung in Abschnitt 3
FUNDAMENTAL_A = None    # Grundschwingungsreststrom in A (None = Median von √(I_w² + I_v²) der Versuche)
LIMITS        = ()      # Grenzwerte des Reststroms in A, z. B. (I_1, I_2)
PQ_SAFETY_FACTOR = 1.0  # Faktor auf das Quantil der PQ-Messung (1.0 = keiner)
PQ_FACTORS    = None    # Faktor auf das Quantil nach Dauer der PQ-Messung {Mindesttage: Faktor} (None = PQ_SAFETY_FACTOR)

## 2 · Kalibrierung mit den Erdschlussversuchen

Je Versuch und Ordnung das **B-Verfahren**: Z_s = 3·p_s·ΔU_s/I (s = Null-, Mit-, Gegensystem; ΔU = Änderung der
Sammelschienenspannung im Fehler, I = Fehlerstrom, p_s = Phasenfaktor des Fehlerleiters). Quelle ist die
Vorfehlerspannung des Fehlerleiters, U_q = −Σ p_s·U_pre,s. Bei den Ordnungen von `TWO_SOURCE` trennen Versuche mit
verschiedener Zusatzimpedanz zusätzlich die im Fehler entstehende Quelle U_s ab (`zwei Quellen`); mit nur einer
Zusatzimpedanz gilt der gemessene Strom für jede Fehlerimpedanz (`konstant`). Mit Spulenströmen in den Versuchen wird
die Admittanz des Netzes ohne die Spulen, Y_0,Netz = (I − I_Spule)/(3·ΔU_0), für die Knotenregel (Abschnitt 4)
behalten.

In [ ]:
for i, e in enumerate(FAULTS + CHECKS):     # nur bekannte Schlüssel (ein vertippter Schlüssel bliebe unbeachtet)
    legacy.check_keys("FAULTS" if i < len(FAULTS) else "CHECKS", e, fc.ENTRY_KEYS + ("current",))

if DEMO:                         # Demo: Kampagne, eine Aufzeichnung der Sammelschienenspannungen und Spulenströme, Ergebnisdateien
    for c in CAMPAIGNS:
        if not (WORK_FOLDER / LAYOUT.campaigns / c).exists():
            demo.make_campaign(str(WORK_FOLDER), c, layout=LAYOUT.language)
    for e in [*FAULTS, *[dict(campaign=c, measurement=t["measurement"], Z=t["Z"]) for c in CAMPAIGNS
                         for t in fc.campaign_tests(WORK_FOLDER, c)]]:
        folder = WORK_FOLDER / LAYOUT.campaigns / e["campaign"]
        cfg = folder / LAYOUT.raw / ("SUB" if LAYOUT.language == "en" else "UW") / (e["measurement"] + ".cfg")
        if e in FAULTS and not cfg.exists():            # aufgezeichneter Erdschluss: Sammelschienenspannungen und Spulenströme
            demo.make_substation_recording(str(cfg), 0.0, drift=demo.polar(1.15, 20), fault_current=False, seed=7,
                                           start=datetime.datetime(2026, 3, 2, 14, 7))
        if not Path(fc.result_path(WORK_FOLDER, e["campaign"], e["measurement"])).exists():
            with open(str(cfg).replace(".cfg", "_truth.json"), encoding="utf-8") as f:
                b = json.load(f)["intervals"]
            r_ = gf.evaluate(gf.open_recording(str(cfg)),
                             gf.Intervals(*(tuple(b[k]) for k in ("steady_state", "pre_fault", "post_fault"))),
                             z_fault_ohm=e.get("Z"))
            r_.save_json(LAYOUT.result_path(folder, e["measurement"]))
if TESTS is None:
    TESTS = [t for c in CAMPAIGNS for t in fc.campaign_tests(WORK_FOLDER, c)]
display(Markdown("**Erdschlussversuche** (`TESTS`)"))
show(pd.DataFrame(TESTS))
MODEL = fc.NetworkModel(NETWORK)
for msg in fc.add_entries(MODEL, WORK_FOLDER, TESTS, "test", orders=ORDERS, min_current_A=MIN_CURRENT_A):
    print("ℹ", msg)
if MODEL.tests:
    show(MODEL.table())
    display(Markdown("**Modell je Ordnung**"))
    show(MODEL.source_table(how=HOW, two_source=TWO_SOURCE))
    for T in MODEL.source_terms(how=HOW, two_source=TWO_SOURCE).values():
        for h in T.hints:
            print("ℹ", h)
else:
    print("Keine Kalibrierung – das Modell braucht mindestens einen Erdschlussversuch mit Fehlerstrom (TESTS).")

## 3 · Prüfung zwischen den Versuchen

Jeder Versuch wird aus seiner eigenen Vorfehlerspannung und Zusatzimpedanz mit dem Modell anderer Versuche
prognostiziert: `alle anderen` = Modell aller übrigen Versuche, `paarweise` = Modell eines anderen Versuchs (jedes
Paar), `Lernkette` = Z_0 eines Versuchs, Z_q aus den Sammelschienenspannungen eines zweiten (wie ein aufgezeichneter
Erdschluss), Prognose für einen dritten. `Abweichung_A` = Prognose − Messung: **negative Werte sind
Unterschätzungen**. Die größte Unterschätzung je Ordnung von `alle anderen` – ein Modell der übrigen Versuche, wie
die Prognose mit allen – ist eine Grundlage für den eigenen Zuschlag `MARGIN_A`; `paarweise` und `Lernkette` zeigen,
wie weit ein einzelner Versuch oder eine Nachführung mit einem aufgezeichneten Erdschluss trägt (mit einem Versuch
behalten die Ordnungen des Nullsystems den Strom dieses Versuchs). Die Zeile `I_H` ist der gesamte
Oberschwingungsanteil (nicht modellierte Ordnungen eingeschlossen).

In [ ]:
CROSS_CHECKS = pd.DataFrame()
if len(MODEL.tests) >= 2:
    modes = ["loo", "pairwise"] + (["chain"] if len(MODEL.tests) >= 3 else [])
    CROSS_CHECKS = pd.concat([MODEL.cross_check(0.0, mode, how=HOW, two_source=TWO_SOURCE) for mode in modes],
                             ignore_index=True)
    summary = (CROSS_CHECKS.assign(order=CROSS_CHECKS["order"].astype(str))
               .groupby(["order", "mode"], sort=False)
               .agg(n=("deviation_A", "size"), deviation_min_A=("deviation_A", "min"),
                    deviation_max_A=("deviation_A", "max"), ratio_min=("ratio", "min"), ratio_max=("ratio", "max"))
               .reset_index())
    show(summary)
    under = CROSS_CHECKS[CROSS_CHECKS["order"] != "I_H"].groupby(["mode", "order"])["deviation_A"].min()
    for mode in [m for m in modes if m in under.index.get_level_values(0)]:   # je Prüfung, ohne Zuschlag
        print(i18n.format_text("Größte Unterschätzung je Ordnung ohne Zuschlag ({}):",
                               i18n.value_label("mode", mode)),
              ", ".join(i18n.format_text("{}: {:.2f} A", m, min(v, 0.0)) for m, v in under[mode].items()))
    display(Markdown("**Alle anderen Versuche (loo)**"))
    show(CROSS_CHECKS[CROSS_CHECKS["mode"] == "loo"])
else:
    print("Die Prüfung braucht mindestens zwei Erdschlussversuche.")

## 4 · Aufgezeichnete Erdschlüsse

Ein an der Sammelschiene aufgezeichneter Erdschluss des Netzes – Leiter-Erde-Spannungen vor und im Fehler,
ausgewertet mit `Messung_auswerten.ipynb` (Vorfehler- und stationärer Bereich; ein Fehlerstrom ist nicht nötig). Mit
Z_0 der Versuche: Z_s = Z_0·p_s·ΔU_s/ΔU_0. Der **Fehlerstrom** des Erdschlusses (`I_geschaetzt_A`): mit Spulenströmen
in der Aufzeichnung und in den Versuchen über die **Knotenregel** des Nullsystems, I_F = 3·Y_0,Netz·ΔU_0 + I_Spule –
bei jeder Ordnung, auch wo im Fehler eine Quelle entsteht –, sonst 3·|ΔU_0|/|Z_0|. Die Ordnungen von `TWO_SOURCE`
werden nicht nachgeführt. Eine Aufzeichnung mit den Leiterströmen des Trafos – oder `transformer=` – liefert
zusätzlich die Vorimpedanz Z_vor und damit das MS-Netz Z_MS. Jeder Erdschluss **prüft dann die Prognose**: Das Modell
aller anderen Erdschlüsse prognostiziert ihn aus seiner eigenen Vorfehlerspannung mit der Fehlerimpedanz 0 (die
Fehlerimpedanz eines natürlichen Erdschlusses ist unbekannt – sein Strom kann nur kleiner sein).

In [ ]:
FAULT_CHECKS = pd.DataFrame()
if FAULTS and not MODEL.tests:
    print("⚠ ein aufgezeichneter Erdschluss braucht Z_0 einer Kalibrierung (Abschnitt 2)")
elif FAULTS:
    for msg in fc.add_entries(MODEL, WORK_FOLDER, FAULTS, "fault", two_source=TWO_SOURCE):
        print("ℹ", msg)
if MODEL.faults:
    tab = MODEL.table()
    show(tab[tab["kind"] == "fault"])
    display(Markdown("**|Z_q| der Erdschlüsse ÷ Median der Versuche**"))
    ref = tab[tab["kind"] == "test"].groupby("order")["Zq_Ohm"].median()
    show(tab[tab["kind"] == "fault"].pivot(index="order", columns="name", values="Zq_Ohm").div(ref, axis=0))
    if any(c.currents for c in MODEL.faults):
        FAULT_CHECKS = MODEL.fault_check(0.0, how=HOW, two_source=TWO_SOURCE)
        display(Markdown("**Prognose (Fehlerimpedanz 0, ohne Zuschlag) ÷ geschätzter Fehlerstrom**"))
        show(FAULT_CHECKS)
else:
    print("Kein aufgezeichneter Erdschluss im Modell (FAULTS).")

## 5 · Prüfung mit weiteren Messungen

Eine Messung, die nicht Teil des Modells ist – z. B. ein Erdschlussversuch an einem Ort im Netz: die
Sammelschienenspannungen der UW-Aufzeichnung (`measurement`), der Fehlerstrom der Aufzeichnung am Fehlerort
(`current`, Beträge je Ordnung) und die Leitung bis zum Fehlerort (`line`). Das Modell prognostiziert den Fehlerstrom
aus der Vorfehlerspannung der Sammelschiene, der Fehlerimpedanz `Z` und der Leitung, Z_q + (2·Z_1L + Z_0L)/3 + Z_F. Für
einen Eintrag mit Leitung messen die Sammelschienenspannungen desselben Fehlers auch die Leitung je Ordnung
(`MODEL.line_from_fault`): Z_qL,m = −U_L,SS,m/I_F,m − Z_F mit dem Fehlerstrom aus der Sammelschiene (Knotenregel, sonst
3·ΔU_0/Z_0 bei den Ordnungen ohne Quelle im Fehler). `LINE_MEASURED` enthält die Werte – als `loop=` einer Leitung geben
sie der Prognose eigene Werte je Ordnung.

In [ ]:
VERIFY = pd.DataFrame()
LINE_MEASURED, LINE_TABLE = {}, pd.DataFrame()
if CHECKS and MODEL.tests:
    parts, rows = [], []
    for e in CHECKS:
        try:
            fd = fc.check_data(WORK_FOLDER, e)
            parts.append(MODEL.verify(fd, 0.0, how=HOW, two_source=TWO_SOURCE))
            if fd.line is not None:          # die Leitung je Ordnung aus den Sammelschienenspannungen desselben Fehlers
                with warnings.catch_warnings(record=True) as caught:
                    warnings.simplefilter("always")
                    z = LINE_MEASURED[fd.name] = MODEL.line_from_fault(fd, fd.Z_F or 0.0, two_source=TWO_SOURCE)
                for w in caught:
                    print("ℹ", w.message)
                for m, v in sorted(z.items()):
                    g = fd.line.Zq(m)
                    rows.append(dict(target=fd.name, order=m, line_values=fd.line.source(m), Z_line_Ohm=abs(g),
                                     R_measured_Ohm=v.real, X_measured_Ohm=v.imag, Z_measured_Ohm=abs(v),
                                     ratio=abs(v) / abs(g) if abs(g) else float("nan")))
        except (OSError, ValueError) as ex:
            print("⚠", ex)
    if parts:
        VERIFY = pd.concat(parts, ignore_index=True)
        show(VERIFY)
    if rows:
        LINE_TABLE = pd.DataFrame(rows)
        display(Markdown("**Leitung je Ordnung: angegeben (`line`) und aus den Sammelschienenspannungen des Fehlers gemessen (`LINE_MEASURED`)**"))
        show(LINE_TABLE)
else:
    print("ℹ Keine weiteren Messungen (CHECKS).")

## 6 · Prognose des Oberschwingungsreststroms

Oberschwingungsspannungen der Sammelschiene je Ordnung (Leiter-Erde, die größte der drei Phasen): ein Wert je Ordnung
– z. B. das 95-%-Quantil einer Netzqualitätsmessung – oder eine Zeitreihe (CSV-Datei, eine Spalte je Ordnung wie
`U_H5`; die letzte Zahl eines Spaltennamens ist die Ordnung). Je Ordnung |I_F,m| = (|U_m| + |U_s,m|)/|Z_q,m + Z_qL,m +
Z_F| + Zuschlag (Z_qL = 0 an der Sammelschiene, (2·Z_1L + Z_0L)/3 hinter `LINE`), dann I_H = √(Σ I_F,m² + I_Rest²)
mit I_Rest = Oberschwingungsanteil der Versuche außerhalb der Prognoseordnungen und I_RES = √(I_1² + I_H²). Eine
Zeitreihe ergibt das Quantil `QUANTILE` der Zeitreihe von I_H und I_RES. Mit `PQ_FILES` erhalten die Ordnungen des Mit-
oder Gegensystems das Quantil der PQ-Messung (mal Faktor aus `PQ_FACTORS`); das Nullsystem (3, 9, 15 …) – es entsteht
im Erdschluss – und die geraden Ordnungen behalten die Vorfehlerspannung der Versuche (nicht skaliert). Eine PQ-Messung
kürzer als jede Dauer von `PQ_FACTORS` (lückenlose Werte) ist nicht gültig und liefert keine Spannungen.

In [ ]:
FORECAST = FORECAST_TABLE = None
PQ = None
if PQ_FILES:                           # PQ-Messung der Sammelschiene: Quantil je Ordnung × Faktor oder die 10-min-Werte
    files = [WORK_FOLDER / p for p in ([PQ_FILES] if isinstance(PQ_FILES, (str, Path)) else PQ_FILES)]
    PQ = pq.assess(pq.read(files, timezone=PQ_TIMEZONE), quantile=QUANTILE,
                   **(dict(factors=PQ_FACTORS) if PQ_FACTORS else dict(factor=PQ_SAFETY_FACTOR)))
    print(PQ.summary())
    for h in PQ.hints:
        print("ℹ", h)
    if not PQ.valid:
        print("⚠ Die PQ-Messung ist für die Bewertung nicht gültig (lückenlose Werte kürzer als jede Dauer von PQ_FACTORS) – keine Spannungen daraus; VOLTAGES aus Abschnitt 1 bleibt.")
    else:
        VOLTAGES, VOLTAGE_REFERENCE = (PQ.series() if PQ_SERIES else PQ.voltages()), PQ.reference
        # Nullsystem (entsteht im Erdschluss) und gerade Ordnungen: nicht skaliert – die Vorfehlerspannung der Versuche
        KEPT = ({m: u for m, u in MODEL.test_voltages().items() if gf.formulas.harmonic_category(m) in ("zero", "even")}
                if VOLTAGE_REFERENCE == "LE" else {})
        for m, u in KEPT.items():
            if PQ_SERIES:
                for df in VOLTAGES.values():
                    df[m] = u
            else:
                VOLTAGES[m] = u
        if KEPT:
            print(i18n.format_text("Nullsystem und gerade Ordnungen – Vorfehlerspannung der Versuche (Median, nicht skaliert): {}",
                                   ", ".join(i18n.format_text("{}: {:.1f} V", m, u) for m, u in KEPT.items())))
        if not PQ_SERIES:
            print(i18n.format_text("VOLTAGES aus der PQ-Messung (Quantil × {:g}, Bezug {}): {}", PQ.factor,
                                   PQ.reference, ", ".join(i18n.format_text("{}: {:.1f} V", m, u) for m, u in VOLTAGES.items()
                                                           if m not in KEPT)))
if VOLTAGES is None:
    print("ℹ Prognose übersprungen – VOLTAGES ist nicht gesetzt (Abschnitt 1).")
elif not MODEL.calibrations:
    print("ℹ Prognose übersprungen – das Modell enthält keine Kalibrierung (Abschnitt 2).")
else:
    U = pd.read_csv(VOLTAGES, **CSV_OPTIONS) if isinstance(VOLTAGES, (str, Path)) else VOLTAGES
    places = [(None, "Sammelschiene")] + ([(LINE, LINE.get("name") or "LINE")] if LINE else [])
    FORECASTS, tables = {}, []
    for line, place in places:
        f = FORECASTS[place] = MODEL.forecast(U, Z_F=Z_F, margin_A=MARGIN_A, how=HOW, two_source=TWO_SOURCE,
                                              reference=VOLTAGE_REFERENCE, fundamental_A=FUNDAMENTAL_A, line=line)
        series = len(f.currents) > 1
        tab = f.table(QUANTILE if series else None)
        tab.insert(0, "location", place)
        tables.append(tab)
        display(Markdown("**%s**" % place))
        show(tab)
        for h in f.hints:
            print("ℹ", h)
        value = tab.set_index("order")["I_forecast_A"]
        print(i18n.format_text("Prognose{}: I_H = {:.2f} A, I_RES = {:.2f} A (Fehlerimpedanz {:g} Ω)",
                               i18n.format_text(" (Quantil {:g} von {} Zeitschritten)",
                                                QUANTILE, len(f.currents)) if series else "",
                               value["I_H"], value["I_RES"], abs(complex(Z_F))))
        if LIMITS:
            for limit in LIMITS:
                print(i18n.format_text("I_RES = {:.2f} A {} {:g} A", value["I_RES"],
                                       "≤" if value["I_RES"] <= limit else ">", limit))
    FORECAST = FORECASTS[places[0][1]]                   # an der Sammelschiene
    FORECAST_TABLE = pd.concat(tables, ignore_index=True)
    if not MARGIN_A:
        print("ℹ Ohne Zuschlag – MARGIN_A ist nicht gesetzt (Abschnitt 1); Abschnitt 3 zeigt die Abweichungen des Modells.")
    if not LIMITS:
        print("ℹ Kein Vergleich mit Grenzwerten – LIMITS ist nicht gesetzt (Abschnitt 1).")

## 7 · Modell speichern und Tabellen exportieren

In [ ]:
OUT = Path(fc.forecast_folder(WORK_FOLDER, NETWORK))
if MODEL.calibrations:
    path = OUT / (report.slug(NETWORK) + "_netzmodell.json")
    MODEL.save(str(path))
    TABLES = {"forecast_calibrations": MODEL.table(),
              "forecast_model": MODEL.source_table(how=HOW, two_source=TWO_SOURCE),
              "forecast_checks": pd.concat([CROSS_CHECKS, VERIFY], ignore_index=True),
              "forecast_faults": FAULT_CHECKS, "forecast": FORECAST_TABLE}
    paths = report.export_tables({k: v for k, v in TABLES.items() if v is not None and len(v)}, str(OUT),
                                 prefix=report.slug(NETWORK) + "_", formats=("csv", "md"))
    print("Modelldatei:", path)
    print("%d Tabellen-Dateien in %s" % (len(paths), OUT))
    # später: fc.NetworkModel.load(path) – Erdschlüsse mit add_fault ergänzen, Prognose mit forecast